<a href="https://colab.research.google.com/github/buiswrld/ecg-leads/blob/main/MSNS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Download and Unzip the PTBXL dataset

### Load python packages

In [1]:
import pandas as pd
import numpy as np
import wfdb
import ast
import gdown
import zipfile
import os
from pathlib import Path
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from resnet1d import ResNet1D

### Download and unzip Google Drive Files

In [ ]:
# Configuration
file_id = "1f4sk1pCJ6SKK8M-TRpEhEiVqvLTHV80p"
output_zip = "download_file.zip"
extract_folder = "unzipped_files"
# Download the Google Drive file
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, output_zip, quiet = False)
# Unzip the zip file
os.makedirs(extract_folder, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

### Access unzipped datasets

In [7]:
# Configuration
# Replace path with your own path
path = '.\\unzipped_files\\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1\\'
sampling_rate=100

## Step 2: Load and Aggregate Metadata

In [5]:
# Functions
def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

In [8]:
# Load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

In [9]:
# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

In [10]:
# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [11]:
# Save the processed metadata to a new CSV file
Y.to_csv("processed_ptbxl_metadata.csv")

## Step 3: Load raw ECG data

In [ ]:
# Functions
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

In [6]:
# Load raw signal data: sampling_rate=100 or 500
X = load_raw_data(Y, sampling_rate, path)

In [7]:
# Save X numpy.ndarray object
np.save("X_numpy_ndarray.npy", X)

## Step 4: Define PTBXLDataset class

In [2]:
class PTBXLDataset(Dataset):
    def __init__(self, csv_path, data_root, split="train", leads=None, mlb=None):
        """
        csv_path: path to 'processed_ptbxl_metadata.csv'
        data_root: folder directory containing 'X_numpy_ndarray.npy'
        split: "train", "val", or "test"
        leads: optional list of lead names, e.g. ["I", "II", "V1"]
        """
        self.split = split
        
        # 1. Load the pre-processed metadata CSV created in Step 2
        df = pd.read_csv(csv_path, index_col='ecg_id')
        
        # Convert string representations of lists back to actual Python lists
        df['diagnostic_superclass'] = df['diagnostic_superclass'].apply(lambda x: ast.literal_eval(x))

        # 2. Filter the index using the recommended stratification folds
        if split == "train":
            mask = (df.strat_fold <= 8).values
        elif split == "val":
            mask = (df.strat_fold == 9).values
        elif split == "test":
            mask = (df.strat_fold == 10).values
        else:
            raise ValueError("Split must be 'train', 'val', or 'test'")
            
        # Keep only the rows for this specific split
        self.df = df[mask].reset_index(drop=True)
        
        # 3. Load the pre-saved numpy matrix from Step 3 and slice it with the exact same mask
        full_X = np.load(os.path.join(data_root, "X_numpy_ndarray.npy"))
        self.X = full_X[mask]
        
        # 4. Handle optional lead selection filtering
        self.lead_map = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
        self.lead_indices = [self.lead_map.index(l) for l in leads] if leads else None

        # 5. Set up the Label Encoder to convert text to numeric vectors
        # If classes are provided (for val/test), use them. Otherwise, fit to the training labels.
        if mlb is None:
            # If no mlb is provided (Training set), create a new one and fit it
            self.mlb = MultiLabelBinarizer()
            self.encoded_labels = self.mlb.fit_transform(self.df['diagnostic_superclass'])
        else:
            # If a pre-fitted mlb is passed (Val/Test sets), reuse it exactly
            self.mlb = mlb
            self.encoded_labels = self.mlb.transform(self.df['diagnostic_superclass'])

    def __len__(self):
        # Return the exact number of rows in this data split
        return len(self.df)

    def __getitem__(self, index):
        # 1. Grab the matrix slice from memory
        ecg_signal = self.X[index]  # Shape: (1000, 12)
        
        # 2. Filter out specific leads if requested
        if self.lead_indices is not None:
            ecg_signal = ecg_signal[:, self.lead_indices]
            
        # 3. Convert to tensor and flip shape to match PyTorch expectations: (channels, time_steps)
        ecg = torch.tensor(ecg_signal, dtype=torch.float32).transpose(0, 1)
        
        # 4. Extract the numeric binary tensor label instead of a text list
        label = torch.tensor(self.encoded_labels[index], dtype=torch.float32)

        return {
            "ecg": ecg,
            "label": label
        }

In [ ]:
# Generate datasets for each split
# 1. Initialize your training dataset
#train_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="train")

# 2. validation and test datesets
#val_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="val", mlb=train_dataset.mlb)
#test_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="test", mlb=train_dataset.mlb)

## Step 5: Build deep learning model

In [3]:
"""
Goal:
Connect the dataset, dataloaders, model, loss function, optimizer, and trainer.

This file is intentionally incomplete. Find active tasks marked by #TODO and fill in the blanks below them.
Some parts may be omitted or added depending on necessity. You can add more code if you think it is necessary.
"""

import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from metrics import compute_classification_metrics

# TODO: Import your dataset class.
# Here, we also need to import a ResNet class, but we leave this exercise for another time, temporarily.
# from resnet1d import ResNet1D

# ------------------------------------------------------------
# Basic settings
# ------------------------------------------------------------

PARAMS = {
    # TODO: Fill in the paths corresponding to where the CSV and data are located for our dataset.
    # Since we are using Colab, make sure to mount your Google Drive at the top and update the paths accordingly.
    "csv_path": "processed_ptbxl_metadata.csv",
    "data_root": ".",

    # None means use all 12 ECG leads.
    # Later, we can replace this with a smaller list of selected leads.
    "leads": None,

    "batch_size": 32,
    "num_workers": 0,
    "learning_rate": 1e-3,
    "max_epochs": 10,

    # PTB-XL diagnostic superclass labels:
    # NORM, MI, STTC, CD, HYP
    "num_classes": 5,
}

# ------------------------------------------------------------
# Training task
# ------------------------------------------------------------

class ECGClassificationTask(pl.LightningModule):
    """
    Defines how the model is trained, validated, and tested.

    This section should include:
    - the model
      - This is where you will define the architecture of your model.
      - We will be using ResNet, but for now, leave this as a placeholder.

    - the loss function
      - This is where you will define the loss function used to train the model.
      - The "loss function" is another way to say "objective function", the function we want to minimize to make the model better.
      - For a multi-label classification problem, this is typically binary cross-entropy loss.
      - In PyTorch, BCEWithLogitsLoss is usually the correct choice when the model outputs raw logits.

    - the training step
      - This is where you will define what happens during one step of training.
      - Training is the process of updating the model's parameters to minimize the loss on the training data.
      - This typically includes getting a batch of data, running it through the model, computing the loss, and logging the loss.

    - the validation step
      - This is where you will define what happens during one step of validation.
      - Validation is used to check how the model is doing on unseen data during training, and to tune hyperparameters.
      - It is not used to evaluate the final model.
      - This typically includes getting a batch of data, running it through the model, computing the loss, and logging the loss.

    - the test step
      - This is where you will define what happens during one step of testing.
      - Testing is similar to validation, but it is used to evaluate the final model after training is complete.
      - This typically includes getting a batch of data, running it through the model, computing the loss, and logging the loss.

    - the optimizer
      - This is where you will define which optimizer to use for training the model.
      - An optimizer's job is to update the model's parameters based on the computed gradients to minimize the loss.
      - This typically includes creating an optimizer, like Adam or SGD, and telling Lightning to use it.
    """

    def __init__(self, num_leads, num_classes, learning_rate):
        super().__init__()
        self.save_hyperparameters()

        self.learning_rate = learning_rate

        # Initialize the model.
        # # For now, leave this as a placeholder. Later, we will replace this with a ResNet architecture, which we will define ourselves in another file. This file will not train until you replace None with a real model.
        #self.model = nn.Sequential(
        #    nn.Flatten(),
        #    nn.Linear(num_leads * 1000, num_classes)
        #)
        from resnet1d import ResNet1D

        self.model = ResNet1D(
            in_channels=num_leads,
            num_classes=num_classes
        )

        # TODO: Initialize the loss function.
        # 1. What do you think is a good loss function for multi-label classification?
        # 2. Should the loss expect probabilities or raw logits?
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.validation_outputs = []
        self.test_outputs = []

    def forward(self, ecg):
        """
        Goal:
        Define how the model produces output from input ECGs.
        """

        # TODO: Run the ECG through the model.
        # Call self.model on the input ECG to get the model's output.
        # 1. What do you think the shape of the output should be?   (32, 5)
        # 2. Should you apply an activation function here? No, because BCEWithLogitsLoss expects raw logits as input, and it will apply the sigmoid function internally.
        # Hint: Check what BCEWithLogitsLoss expects as input.

        # Returns: raw logits of shape (batch_size, num_classes).
        # # Note/Hint: Do not apply sigmoid here if using BCEWithLogitsLoss. Sigmoid can be used later when converting logits into probabilities for evaluation or interpretation.
        return self.model(ecg)

    def training_step(self, batch, batch_idx):
        """
        Goal:
        Use one batch to compute training loss.
        """

        # TODO: Get ECGs and labels from the batch.
        # 1. How can you get the ECGs and labels from the batch?
        # 2. What do you think their shapes will be? Can we check?
        ecgs = batch["ecg"]      # Shape: (batch_size, num_leads, 1000)
        labels = batch["label"]  # Shape: (batch_size, num_classes)

        # TODO: Get model outputs.
        # Call self.forward on the input ECGs to get the model's raw logits.
        logits = self.forward(ecgs)  # Shape: (batch_size, num_classes)

        # TODO: Compute loss.
        # Call the loss function on the model's logits and the true labels to compute the loss.
        loss = self.loss_fn(logits, labels)

        # TODO: Log training loss.
        # Use self.log to log the training loss. This will allow us to log it and track it over time.
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)   

        # TODO: Return loss.
        # The training step should return the loss, which will be used by the optimizer to update the model's parameters.
        # Returns: the training loss.
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Goal:
        Check how the model performs on validation data.
        """

        # TODO: Get ECGs and labels from the batch.
        # 1. How can you get the ECGs and labels from the batch?
        # 2. What do you think their shapes will be? Can we check?
        ecgs = batch["ecg"]
        labels = batch["label"]

        # TODO: Get model outputs.
        # Call self.forward on the input ECGs to get the model's raw logits.
        logits = self.forward(ecgs)
        
        # TODO: Compute validation loss.
        # Call the loss function on the model's logits and the true labels to compute the loss.
        loss = self.loss_fn(logits, labels)
        self.validation_outputs.append((logits.detach(), labels.detach()))

        # TODO: Log validation loss.
        # Use self.log to log the validation loss. This will allow us to log it and track it over time.
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        # Returns: the validation loss.
        return loss

    def test_step(self, batch, batch_idx):
        """
        Goal:
        Evaluate the final model on test data.
        """
        ecgs = batch["ecg"]
        labels = batch["label"]
        
        logits = self.forward(ecgs)
        loss = self.loss_fn(logits, labels)
        self.test_outputs.append((logits.detach(), labels.detach()))
        
        # Log final test metric evaluation
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        # TODO: Write a test step.
        # This will be similar to the validation step, but it will be used to evaluate the final model after training is complete.
        # See the validation step for guidance.
        
        # Returns: the test loss.
        return loss

    def on_validation_epoch_end(self):
        if not self.validation_outputs:
            return

        logits = torch.cat([item[0] for item in self.validation_outputs], dim=0)
        labels = torch.cat([item[1] for item in self.validation_outputs], dim=0)
        metrics = compute_classification_metrics(logits, labels)
        self.log("val_auroc", metrics["auroc"], prog_bar=True, logger=True)
        self.log("val_auprc", metrics["auprc"], prog_bar=True, logger=True)
        self.log("val_f1", metrics["f1"], prog_bar=True, logger=True)
        self.validation_outputs.clear()

    def on_test_epoch_end(self):
        if not self.test_outputs:
            return

        logits = torch.cat([item[0] for item in self.test_outputs], dim=0)
        labels = torch.cat([item[1] for item in self.test_outputs], dim=0)
        metrics = compute_classification_metrics(logits, labels)
        self.log("test_auroc", metrics["auroc"], prog_bar=True, logger=True)
        self.log("test_auprc", metrics["auprc"], prog_bar=True, logger=True)
        self.log("test_f1", metrics["f1"], prog_bar=True, logger=True)
        self.test_outputs.clear()

    def configure_optimizers(self):
        """
        Goal:
        Tell Lightning which optimizer to use.
        """
               
        # TODO: Create and return an optimizer.
        # 1. Which optimizer do you think is a good choice for training this model? Why?
        # 2. How do you create an optimizer in PyTorch?
        # 3. What parameters do you need to pass to it?
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)

        # Returns: the optimizer.
        return optimizer


# ------------------------------------------------------------
# Data setup
# ------------------------------------------------------------

def build_dataloaders(params):
    """
    Goal:
    Create train, validation, and test dataloaders.

    The dataset handles loading ECGs and labels.
    The dataloader handles batching and shuffling.

    Important:
    The training dataset should create or fit the label binarizer.
    The validation and test datasets should reuse the same label binarizer.
    This keeps the label order consistent across train, validation, and test.
    """

    # TODO: Create the training dataset.
    # 1. What parameters do you need to create the dataset?
    # 2. Check the PTBXLDataset __init__ method to see what arguments are required.
    train_dataset = PTBXLDataset(
        csv_path=params["csv_path"],
        data_root=params["data_root"],
        split="train",
        leads=params["leads"],
        mlb=None
    )
    fitted_mlb = train_dataset.mlb
    
    # TODO: Create the validation dataset.
    # See the training dataset for guidance.
    # This will be similar, but it will use the validation data instead of the training data.
    # Make sure it uses the same label binarizer as the training dataset.
    val_dataset = PTBXLDataset(
        csv_path=params["csv_path"],
        data_root=params["data_root"],
        split="val",
        leads=params["leads"],
        mlb=fitted_mlb
    )

    # TODO: Create the test dataset.
    # See the training dataset for guidance.
    # This will be similar, but it will use the test data instead of the training data.
    # Make sure it uses the same label binarizer as the training dataset.
    test_dataset = PTBXLDataset(
        csv_path=params["csv_path"],
        data_root=params["data_root"],
        split="test",
        leads=params["leads"],
        mlb=fitted_mlb
    )

    # TODO: Create the training dataloader.
    # 1. What parameters do you need to create the dataloader?
    # 2. Check the DataLoader documentation or examples to decide which arguments are needed.
    # 3. Do we want to shuffle the training data? What about the validation and test data? Why or why not?
    train_loader = DataLoader(
        train_dataset, 
        batch_size=params["batch_size"], 
        shuffle=True, 
        num_workers=params["num_workers"]
    )

    # TODO: Create the validation dataloader.
    # See the training dataloader for guidance.
    # This will be similar, but it will use the validation dataset instead of the training dataset.
    val_loader = DataLoader(
        val_dataset, 
        batch_size=params["batch_size"], 
        shuffle=False, 
        num_workers=params["num_workers"]
    )

    # TODO: Create the test dataloader.
    # See the training dataloader for guidance.
    # This will be similar, but it will use the test dataset instead of the training dataset.
    test_loader = DataLoader(
        test_dataset, 
        batch_size=params["batch_size"], 
        shuffle=False, 
        num_workers=params["num_workers"]
    )

    # TODO: Return the dataloaders and anything else needed later.
    # Returns: the training dataloader, the validation dataloader, and the test dataloader.
    return train_loader, val_loader, test_loader

# ------------------------------------------------------------
# Lead helper
# ------------------------------------------------------------

def get_num_leads(params):
    """
    Goal:
    Figure out how many ECG leads the model should expect.
    """
    
    # TODO: Return 12 if using all leads. Otherwise, return the number of selected leads.
    # 1. How can we tell if we are using all leads or not?
    # 2. Is there a parameter we can check in the PARAMS dictionary? If not, how can we modify the PARAMS dictionary to include this information?
    # Returns: the number of leads.
    if params["leads"] is None:
        return 12
    else:
        return len(params["leads"])

W0603 12:15:05.941000 1148 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [ ]:
if __name__ == "__main__":
    # Dynamically extract leads count
    num_leads = get_num_leads(PARAMS)
    
    # Build your loaders
    train_loader, val_loader, test_loader = build_dataloaders(PARAMS)
    
    # Initialize the Lightning Module
    task = ECGClassificationTask(
        num_leads=num_leads,
        num_classes=PARAMS["num_classes"],
        learning_rate=PARAMS["learning_rate"]
    )
    
    # Initialize standard PyTorch Lightning Trainer architecture pipeline
    trainer = pl.Trainer(
        max_epochs=PARAMS["max_epochs"],
        accelerator="auto", # Automatically assigns GPU/TPU/CPU resources on Colab
    )
    
    print("Ready to run trainer.fit(task, train_loader, val_loader)")

In [ ]:
##### Model training and Validation
trainer = pl.Trainer(
    max_epochs=PARAMS["max_epochs"],
    accelerator="cpu"
)
trainer.fit(task, train_loader, val_loader)

In [ ]:
##### Model testing
trainer.test(task, test_loader)